In [1]:
import leafmap

In [2]:
m = leafmap.Map(center=[-6.498989966971687, 106.69952474066585], zoom=18)
m.add_basemap('SATELLITE')

m

Map(center=[-6.498989966971687, 106.69952474066585], controls=(ZoomControl(options=['position', 'zoom_in_text'…

In [3]:
if m.user_roi_bounds() is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [106.86633732006248, -6.3102817937379865, 106.84633732006248, -6.2902817937379865]

In [4]:
import os
if os.path.exists("image_test") and os.path.isdir("image_test"):
    pass
else:
    os.makedirs("image_test")
image = "image_test/satellite.tif"

In [5]:
from utility.leafmap_ultility import tms_to_geotiff
tms_to_geotiff(output=image, bbox=bbox, zoom=18, source="Satellite", overwrite=True)

Downloaded image 01/12
Downloaded image 02/12
Downloaded image 03/12
Downloaded image 04/12
Downloaded image 05/12
Downloaded image 06/12
Downloaded image 07/12
Downloaded image 08/12
Downloaded image 09/12
Downloaded image 10/12
Downloaded image 11/12
Downloaded image 12/12
Saving GeoTIFF. Please wait...
Image saved to image_test/satellite.tif


In [6]:
import cv2

In [7]:
import ultralytics

In [8]:
ultralytics.checks()

Ultralytics 8.3.83  Python-3.11.11 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Setup complete  (16 CPUs, 27.8 GB RAM, 836.3/951.7 GB disk)


In [ ]:
#Optional Enchance jika ingin meningkatkan kualitas gambar tiff sebelum deteksi

import numpy as np

def enhance_tiff_detection(image_path, scale_factor=2):
    # Baca gambar
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    
    # Upscale
    upscaled = cv2.resize(img, None, fx=scale_factor, fy=scale_factor, 
                         interpolation=cv2.INTER_CUBIC)
    
    # Sharpening
    kernel = np.array([[-1,-1,-1],
                      [-1, 9,-1],
                      [-1,-1,-1]])
    sharpened = cv2.filter2D(upscaled, -1, kernel)
    
    # Normalisasi jika diperlukan
    if img.dtype == np.uint16:
        sharpened = cv2.normalize(sharpened, None, 0, 65535, cv2.NORM_MINMAX)
    else:
        sharpened = cv2.normalize(sharpened, None, 0, 255, cv2.NORM_MINMAX)
    
    return sharpened

In [ ]:
#Optional Enchance jika ingin meningkatkan kualitas gambar tiff sebelum deteksi
rescale_image = enhance_tiff_detection(image)
cv2.imwrite(image, rescale_image)

True

In [11]:
# Load an image
img_path = image
img = cv2.imread(img_path)

In [12]:
from ultralytics import YOLO

In [13]:
# Load YOLO OBB model
model = YOLO("best.pt")

In [14]:
yolo_results = model(img)


0: 512x640 52.6ms
Speed: 3.6ms preprocess, 52.6ms inference, 89.9ms postprocess per image at shape (1, 3, 512, 640)


In [15]:
for result in yolo_results:
    obb = result.obb
    print("obb", obb)

obb ultralytics.engine.results.OBB object with attributes:

cls: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [16]:
from ultralytics import SAM

In [17]:
sam_model = SAM("sam2.1_b.pt")

In [18]:
sam_model.info()

Model summary: 403 layers, 80,850,178 parameters, 80,850,178 gradients


(403, 80850178, 80850178, 0.0)

In [19]:
xyxy = obb.xyxy
# Convert to Python list
xyxy_all = xyxy.tolist()
print("all xyxy", xyxy_all)

all xyxy [[1337.86083984375, 770.665283203125, 1384.033447265625, 818.93896484375], [1638.3525390625, 756.4826049804688, 1706.217041015625, 819.4036254882812], [1566.322509765625, 1015.582763671875, 1623.579345703125, 1075.573974609375], [1192.6116943359375, 490.8839416503906, 1249.5792236328125, 538.3187255859375], [1355.0889892578125, 696.7006225585938, 1406.3546142578125, 766.6417846679688], [1573.2799072265625, 897.704345703125, 1628.7701416015625, 938.4827880859375], [1678.4853515625, 567.4365844726562, 1742.528076171875, 658.9755249023438], [1314.493896484375, 276.5266418457031, 1388.6650390625, 358.4305114746094], [1572.9815673828125, 38.761627197265625, 1608.8795166015625, 97.02574157714844], [1471.84814453125, 464.534423828125, 1538.5830078125, 529.650634765625], [1732.2333984375, 540.6509399414062, 1804.522705078125, 640.2627563476562], [1051.2459716796875, 517.08544921875, 1117.4173583984375, 555.6353759765625], [886.7365112304688, 642.2106323242188, 948.8334350585938, 716.0

In [20]:
sam_results = sam_model(img, bboxes=xyxy_all)


0: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 1 22, 1 23, 1 24, 1 25, 1 26, 1 27, 1 28, 1 29, 1 30, 1 31, 1 32, 1 33, 1 34, 1 35, 1 36, 1 37, 1 38, 1 39, 1 40, 1 41, 1 42, 1 43, 1 44, 1 45, 1 46, 1 47, 1 48, 1 49, 1 50, 1 51, 1 52, 1 53, 1 54, 1 55, 1 56, 1 57, 1 58, 1 59, 1 60, 1 61, 1 62, 1 63, 1 64, 1 65, 1 66, 1 67, 1 68, 1 69, 1 70, 1 71, 1 72, 1 73, 1 74, 1 75, 1 76, 1 77, 1 78, 1 79, 1 80, 1 81, 1 82, 1 83, 1 84, 1 85, 1 86, 1 87, 1 88, 1 89, 1 90, 1 91, 1 92, 1 93, 1 94, 1 95, 1 96, 1 97, 1 98, 1 99, 1 100, 1 101, 1 102, 1 103, 1 104, 1 105, 1 106, 1 107, 1 108, 1 109, 1 110, 1 111, 1 112, 1 113, 1 114, 1 115, 1 116, 1 117, 1 118, 1 119, 1 120, 1 121, 1 122, 1 123, 1 124, 1 125, 1 126, 1 127, 1 128, 1 129, 1 130, 1 131, 1 132, 1 133, 1 134, 1 135, 1 136, 1 137, 1 138, 1 139, 1 140, 1 141, 1 142, 1 143, 1 144, 1 145, 1 146, 1 147, 1 148, 1 149, 1 150, 1 151, 1 152, 1 153, 1 154, 1 155, 1 15

In [21]:
for result_sam in sam_results:
    masks = result_sam.masks
    print("masks data", masks.data.shape)

masks data torch.Size([300, 1388, 1826])


In [26]:
import numpy as np

# Copy the original image for overlay
overlay = img.copy()

# Create an empty mask to accumulate all object masks
combined_mask = np.zeros((overlay.shape[0], overlay.shape[1]), dtype=np.uint8)

# Extract and combine all masks
for mask in masks.data:
    mask = mask.cpu().numpy().astype(np.uint8) * 255  # Convert boolean mask to 0-255
    combined_mask = np.maximum(combined_mask, mask)  # Combine masks

# Create a color overlay
colored_mask = np.zeros_like(overlay, dtype=np.uint8)
colored_mask[:, :, 2] = combined_mask  # Apply to the red channel

# Blend the mask with the image
overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)
cv2.imwrite(F"{img_path}_mask.png", overlay)

True

In [27]:
import numpy as np

# Copy the original image for overlay
bw_img = img.copy()

# Create a black-and-white mask (0 = black, 255 = white)
bw_mask = np.zeros((bw_img.shape[0], bw_img.shape[1]), dtype=np.uint8)

# Combine all masks into one binary image
for mask in masks.data:
    mask_np = mask.cpu().numpy().astype(np.uint8) * 255  # Convert to 0-255
    bw_mask = np.maximum(bw_mask, mask_np)  # Merge multiple masks

# Save the binary mask image
cv2.imwrite(F"{img_path}_binary_mask.png", bw_mask)

True

In [29]:
import json
import cv2
import numpy as np
from shapely.geometry import Polygon, mapping

# Load the binary mask
bw_mask = cv2.imread(F"{img_path}_binary_mask.png", cv2.IMREAD_GRAYSCALE)

# Find contours (retrieves outermost boundaries)
contours, _ = cv2.findContours(bw_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Convert contours to GeoJSON format
geojson_features = []
for contour in contours:
    if len(contour) >= 3:  # Need at least 3 points to form a polygon
        polygon = Polygon(contour.squeeze().tolist())
        geojson_features.append({
            "type": "Feature",
            "geometry": mapping(polygon),
            "properties": {}
        })

# Save to a GeoJSON file
geojson_data = {
    "type": "FeatureCollection",
    "features": geojson_features
}

with open(F"{img_path}_mask.geojson", "w") as f:
    json.dump(geojson_data, f, indent=4)